In [2]:
import pandas as pd
import numpy as np
import optuna
import torch
import contextlib
import os
import time
import pickle
import warnings
import random
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator

from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\tweiz\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Data Fetching and preprocessing**

In [ ]:
COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Load both files and combine — we apply our own 60/20/20 split
df_train = pd.read_csv("data/adult/adult.data", header=None, names=COLUMN_NAMES,
                       skipinitialspace=True)
df_test  = pd.read_csv("data/adult/adult.test", header=None, names=COLUMN_NAMES,
                       skipinitialspace=True, skiprows=1)  

# The test file has a trailing dot on labels (e.g. "<=50K."), strip it
df_test["income"] = df_test["income"].str.rstrip(".")

# Combine into one dataframe
df = pd.concat([df_train, df_test], ignore_index=True)

print(df.shape)
print(df.head())

(48842, 15)
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0 

In [4]:
# Check missing values — adult uses "?" as the missing marker
print("'?' counts per column:")
print((df == "?").sum())

print("\nClass distribution:")
print(df["income"].value_counts())
print(df["income"].value_counts(normalize=True).round(3))

#print missing value as percentage
print("\n'?' percentage per column:")
print(((df == "?").sum() / len(df) * 100).round(2))


'?' counts per column:
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
income               0
dtype: int64

Class distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64
income
<=50K    0.761
>50K     0.239
Name: proportion, dtype: float64

'?' percentage per column:
age               0.00
workclass         5.73
fnlwgt            0.00
education         0.00
education-num     0.00
marital-status    0.00
occupation        5.75
relationship      0.00
race              0.00
sex               0.00
capital-gain      0.00
capital-loss      0.00
hours-per-week    0.00
native-country    1.75
income            0.00
dtype: float64


2 columns were dropped as they were uninformative

In [4]:
# Drop uninformative columns upfront 
# fnlwgt: census sampling weight, not a predictive feature
# education: string duplicate of education-num (same info, ordinal encoded)
df = df.drop(columns=["fnlwgt", "education"])

# Binary encode target 
df["income"] = (df["income"] == ">50K").astype(int)

print(df.shape)
print(df.head())

(48842, 13)
   age         workclass  education-num      marital-status  \
0   39         State-gov             13       Never-married   
1   50  Self-emp-not-inc             13  Married-civ-spouse   
2   38           Private              9            Divorced   
3   53           Private              7  Married-civ-spouse   
4   28           Private             13  Married-civ-spouse   

          occupation   relationship   race     sex  capital-gain  \
0       Adm-clerical  Not-in-family  White    Male          2174   
1    Exec-managerial        Husband  White    Male             0   
2  Handlers-cleaners  Not-in-family  White    Male             0   
3  Handlers-cleaners        Husband  Black    Male             0   
4     Prof-specialty           Wife  Black  Female             0   

   capital-loss  hours-per-week native-country  income  
0             0              40  United-States       0  
1             0              13  United-States       0  
2             0              

In [5]:
#group columns by type for later use
CATEGORICAL_COLS = ["workclass", "marital-status", "occupation",
                    "relationship", "race", "sex", "native-country"]
NUMERICAL_COLS   = ["age", "education-num", "capital-gain",
                    "capital-loss", "hours-per-week"]

y = df["income"]
X = df.drop(columns=["income"])

print("Features:", X.columns.tolist())
print("Target distribution:\n", y.value_counts())

#print number of numerical and categorical features
print(f"Number of numerical features: {len(NUMERICAL_COLS)}")
print(f"Number of categorical features: {len(CATEGORICAL_COLS)}")

Features: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
Target distribution:
 income
0    37155
1    11687
Name: count, dtype: int64
Number of numerical features: 5
Number of categorical features: 7


Check for cardinality

In [7]:
# Check cardinality of categorical columns
print("Categorical column cardinality:")
for col in CATEGORICAL_COLS:
    print(f"  {col}: {X[col].nunique()} unique values")

#print average cardinality of categorical features
avg_cardinality = np.mean([X[col].nunique() for col in CATEGORICAL_COLS])
print(f"Average cardinality of categorical features: {avg_cardinality:.1f}")

Categorical column cardinality:
  workclass: 9 unique values
  marital-status: 7 unique values
  occupation: 15 unique values
  relationship: 6 unique values
  race: 5 unique values
  sex: 2 unique values
  native-country: 42 unique values
Average cardinality of categorical features: 12.3


**TabNet**
---
Adult Income contains a mix of numeric and categorical features, with missing values marked as "?" in `workclass`, `occupation`, and `native-country`. 

For categorical features, TabNet requires integer-encoded inputs. It provides native categorical embedding support via cat_idxs and cat_dims, where each category is mapped to a learned dense vector representation. As a result, label encoding (assigning a unique integer to each category) is sufficient, and techniques such as one-hot encoding or target encoding are not required. The "?" missing values are treated as their own category during label encoding and require no additional processing.

For numeric features, StandardScaler normalization is applied, with the scaler fit on the training set only and applied to validation and test sets to prevent data leakage.

The data is split into 60/20/20 train/validation/test sets using stratified sampling to preserve the class imbalance (~76% ≤50K, ~24% >50K). 

In [ ]:
# Label-encode categoricals — "?" becomes its own category naturally
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# 60/20/20 split (stratified to preserve class balance)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0, stratify=y_temp)

# Scale only numeric columns — fit on train only
scaler = StandardScaler()
X_train[NUMERICAL_COLS] = scaler.fit_transform(X_train[NUMERICAL_COLS])
X_val[NUMERICAL_COLS]   = scaler.transform(X_val[NUMERICAL_COLS])
X_test[NUMERICAL_COLS]  = scaler.transform(X_test[NUMERICAL_COLS])

# Build cat_idxs and cat_dims for TabNet
cat_idxs = [X_train.columns.get_loc(c) for c in CATEGORICAL_COLS]
cat_dims  = [X[c].nunique() for c in CATEGORICAL_COLS]

# Convert to numpy arrays for TabNet
X_train_tab = X_train.values.astype(np.float32)
X_val_tab   = X_val.values.astype(np.float32)
X_test_tab  = X_test.values.astype(np.float32)

y_train_arr = y_train.values
y_val_arr   = y_val.values
y_test_arr  = y_test.values

print("cat_idxs:", cat_idxs)
print("cat_dims:", cat_dims)
print("Train shape:", X_train_tab.shape)

cat_idxs: [1, 3, 4, 5, 6, 7, 11]
cat_dims: [9, 7, 15, 6, 5, 2, 42]
Train shape: (29305, 12)


**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `n_d` / `n_a` (kept equal): controls the width of the decision step
- `n_steps`: number of sequential attention steps
- `lr`: learning rate

In [ ]:
def objective(trial):
    n_da = trial.suggest_int("n_da", 8, 32, step=8)
    params = {
        "n_da": n_da,
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    }

    model = TabNetClassifier(
        n_d=params["n_da"],
        n_a=params["n_da"],
        n_steps=params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        optimizer_params=dict(lr=params["lr"]),
        seed=0,
        verbose=0
    )

    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_tab, y_train_arr,
            eval_set=[(X_val_tab, y_val_arr)],
            eval_metric=["auc"],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )

    preds_proba = model.predict_proba(X_val_tab)[:, 1]
    auc = roc_auc_score(y_val_arr, preds_proba)
    return auc  # maximize

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

Best AUC: 0.9144682210266847
Best params: {'n_da': 16, 'n_steps': 3, 'lr': 0.009917825073986314}


**Final Evaluation**

In [ ]:
# final evaluation across 3 seeds
best_params = study.best_params
seeds = [1, 2, 3]

final_acc, final_auc = [], []

for seed in seeds:
    model = TabNetClassifier(
        n_d=best_params["n_da"],
        n_a=best_params["n_da"],
        n_steps=best_params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        optimizer_params=dict(lr=best_params["lr"]),
        seed=seed,
        verbose=0
    )

    with open(os.devnull, "w") as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_tab, y_train_arr,
            eval_set=[(X_val_tab, y_val_arr)],
            eval_metric=["auc"],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )

    preds = model.predict(X_test_tab)
    preds_proba = model.predict_proba(X_test_tab)[:, 1]

    final_acc.append(accuracy_score(y_test_arr, preds))
    final_auc.append(roc_auc_score(y_test_arr, preds_proba))

print(f"TabNet Accuracy: {np.mean(final_acc):.4f} ± {np.std(final_acc):.4f}")
print(f"TabNet AUC-ROC:  {np.mean(final_auc):.4f} ± {np.std(final_auc):.4f}")

TabNet Accuracy: 0.8482 ± 0.0006
TabNet AUC-ROC:  0.9023 ± 0.0029


**Inference Cost**

In [ ]:
start = time.time()
_ = model.predict(X_test_tab)
elapsed = time.time() - start

print(f"Inference time per sample: {elapsed/len(X_test_tab)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model)) / 1024:.2f} KB")

Inference time per sample: 0.0107 ms
Model size: 604.21 KB


**Export Results**

In [ ]:
# export obtained results, including feature importance ranking
results = {
    "Model": ["TabNet"],
    "Dataset": ["Adult Income"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc)],
    "Accuracy Std": [np.std(final_acc)],
    "AUC-ROC Mean": [np.mean(final_auc)],
    "AUC-ROC Std": [np.std(final_auc)],
    "Inference Time (ms/sample)": [elapsed / len(X_test_tab) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model)) / 1024],
    "Feature Ranking (most to least important)": [
        str(pd.Series(model.feature_importances_, index=X_train.columns)
            .sort_values(ascending=False)
            .index.tolist())
    ]
}

results_df = pd.DataFrame(results)
results_df.to_csv("results/results_adult_income.csv", index=False)
print(results_df)

    Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0  TabNet  Adult Income  Classification       0.848193      0.000585   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.902286     0.002861                    0.010661       604.207031   

           Feature Ranking (most to least important)  
0  ['education-num', 'relationship', 'age', 'capi...  


**FT-Transformer**
---
The same train/validation/test split is used.

FT-Transformer is implemented via pytorch-tabular, which accepts pandas DataFrames directly and handles categorical embeddings internally through its DataConfig — categorical columns are passed as raw strings and pytorch-tabular learns embeddings for them natively. As a result, no label encoding is required for FT-Transformer, unlike TabNet. The "?" missing values are again treated as their own category and require no further processing.

For numeric features, StandardScaler normalization is similarly applied, with the scaler fit on the training set only and applied to validation and test sets to prevent data leakage. 

As pytorch-tabular operates on DataFrames, the final inputs are reconstructed by combining scaled numeric features with original categorical columns, aligned using the same train/validation/test indices.




In [ ]:
# FT-Transformer needs DataFrames with original string categoricals
# df still has original strings — use split indices to align with the same 60/20/20 split
train_df_ft = df.loc[X_train.index].copy().reset_index(drop=True)
val_df_ft   = df.loc[X_val.index].copy().reset_index(drop=True)
test_df_ft  = df.loc[X_test.index].copy().reset_index(drop=True)

# Apply already-scaled numeric values
for ft_df, split in zip([train_df_ft, val_df_ft, test_df_ft],
                        [X_train, X_val, X_test]):
    ft_df[NUMERICAL_COLS] = split[NUMERICAL_COLS].values



(29305, 13)
age               float64
workclass          object
education-num     float64
marital-status     object
occupation         object
relationship       object
race               object
sex                object
capital-gain      float64
capital-loss      float64
hours-per-week    float64
native-country     object
income              int64
dtype: object


**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `attn_heads`: number of attention heads in each transformer block (4 or 8)
- `num_layers`: number of transformer blocks
- `lr`: learning rate

In [ ]:
def objective_ft(trial):
    # hyperparameters to tune
    attn_heads = trial.suggest_categorical("attn_heads", [4, 8])
    num_layers = trial.suggest_int("num_layers", 2, 6)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    # data config
    data_config = DataConfig(
        target=["income"],
        continuous_cols=NUMERICAL_COLS,
        categorical_cols=CATEGORICAL_COLS
    )

    # trainer config (FIXED: load_best=False since no checkpoints)
    trainer_config = TrainerConfig(
        max_epochs=30,
        early_stopping="valid_loss",
        early_stopping_patience=3,
        checkpoints=None,
        load_best=False,
        progress_bar="none",
        trainer_kwargs={"enable_model_summary": False}
    )

    optimizer_config = OptimizerConfig()

    # model config
    model_config = FTTransformerConfig(
        task="classification",
        num_attn_blocks=num_layers,
        num_heads=attn_heads,
        learning_rate=lr
    )

    # build model
    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config
    )

    # train
    model_ft.fit(train=train_df_ft, validation=val_df_ft)

    # predict
    preds = model_ft.predict(val_df_ft)

    # safely extract probability column (works for any naming)
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")

    preds_proba = preds[prob_cols[-1]].values

    # compute AUC
    auc = roc_auc_score(val_df_ft["income"].values, preds_proba)

    return auc


# run Optuna
study_ft = optuna.create_study(direction="maximize")
study_ft.optimize(objective_ft, n_trials=50)  

print("Best AUC:", study_ft.best_value)
print("Best params:", study_ft.best_params)

2026-04-08 02:02:18,978 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-08 02:02:18,993 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-08 02:02:19,017 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-08 02:02:19,097 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-08 02:02:19,122 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-08 02:02:19,135 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-08 02:03:19,206 - {pytorch_tabular.tabular_model:690

Best AUC: 0.9136437769196765
Best params: {'attn_heads': 8, 'num_layers': 2, 'lr': 0.00021004866404179186}


**Final Evaluation**

In [ ]:
# final evaluation across 3 seeds
best_params_ft = study_ft.best_params
seeds = [1, 2, 3]

final_acc_ft, final_auc_ft = [], []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    data_config = DataConfig(
        target=["income"],
        continuous_cols=NUMERICAL_COLS,
        categorical_cols=CATEGORICAL_COLS
    )

    trainer_config = TrainerConfig(
        max_epochs=30,
        early_stopping="valid_loss",
        early_stopping_patience=3,
        checkpoints=None,
        load_best=False,
        progress_bar="none",
        seed=seed,
        trainer_kwargs={"enable_model_summary": False}
    )

    optimizer_config = OptimizerConfig()

    model_config = FTTransformerConfig(
        task="classification",
        num_attn_blocks=best_params_ft["num_layers"],
        num_heads=best_params_ft["attn_heads"],
        learning_rate=best_params_ft["lr"]
    )

    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config
    )

    # shuffle training rows to vary batch order across seeds
    train_idx = np.random.permutation(len(train_df_ft))
    train_df_ft_shuffled = train_df_ft.iloc[train_idx].reset_index(drop=True)

    model_ft.fit(train=train_df_ft_shuffled, validation=val_df_ft)

    preds = model_ft.predict(test_df_ft)

    # get positive-class probability
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")

    preds_proba = preds[prob_cols[-1]].values

    # convert probability to class label using 0.5 threshold
    preds_label = (preds_proba >= 0.5).astype(int)

    final_acc_ft.append(accuracy_score(test_df_ft["income"].values, preds_label))
    final_auc_ft.append(roc_auc_score(test_df_ft["income"].values, preds_proba))

print(f"FT-Transformer Accuracy: {np.mean(final_acc_ft):.4f} ± {np.std(final_acc_ft):.4f}")
print(f"FT-Transformer AUC-ROC:  {np.mean(final_auc_ft):.4f} ± {np.std(final_auc_ft):.4f}")

2026-04-08 03:20:29,075 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-08 03:20:29,099 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-08 03:20:29,125 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-08 03:20:29,211 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-08 03:20:29,242 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-08 03:20:29,252 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-08 03:21:45,180 - {pytorch_tabular.tabular_model:690

FT-Transformer Accuracy: 0.8496 ± 0.0017
FT-Transformer AUC-ROC:  0.9088 ± 0.0006


NOTE: 

Since pytorch_tabular internally calls seed_everything(42) regardless of the seed parameter passed to TrainerConfig, we implement an alternative approach: shuffling the training data order with different seeds before training. Even though weight initialization remains locked to seed 42, different batch orderings during training produce results in varying final model weights and predictions.

**Inference Cost**

In [ ]:
# inference cost
start = time.time()
_ = model_ft.predict(test_df_ft)
elapsed_ft = time.time() - start

print(f"Inference time per sample: {elapsed_ft / len(test_df_ft) * 1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_ft)) / 1024:.2f} KB")

Inference time per sample: 0.0597 ms
Model size: 4893.69 KB


**Feature Importance**

In [ ]:
# manual permutation importance for FT-Transformer
from sklearn.metrics import roc_auc_score

X_test_ft = test_df_ft.drop(columns=["income"]).copy()
y_test_ft = test_df_ft["income"].values

def get_positive_proba(model, df):
    preds = model.predict(df)
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")
    return preds[prob_cols[-1]].to_numpy().ravel()

# baseline AUC
baseline_proba = get_positive_proba(model_ft, X_test_ft)
baseline_auc = roc_auc_score(y_test_ft, baseline_proba)

n_repeats = 5
rng = np.random.RandomState(42)

importances_mean = []
importances_all = {}

for col in X_test_ft.columns:
    drops = []

    for _ in range(n_repeats):
        X_perm = X_test_ft.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)

        perm_proba = get_positive_proba(model_ft, X_perm)
        perm_auc = roc_auc_score(y_test_ft, perm_proba)

        drop = baseline_auc - perm_auc
        drops.append(drop)

    importances_all[col] = drops
    importances_mean.append(np.mean(drops))

# convert to ranked series
ft_importance_series = pd.Series(importances_mean, index=X_test_ft.columns).sort_values(ascending=False)

print("Baseline AUC:", baseline_auc)
print("\nFT-Transformer permutation importances (mean AUC drop):")
print(ft_importance_series)

ft_feature_ranking = ft_importance_series.index.tolist()
print("\nFeature ranking (most to least important):")
print(ft_feature_ranking)

Baseline AUC: 0.9094921639505462

FT-Transformer permutation importances (mean AUC drop):
education-num     0.035316
marital-status    0.025369
relationship      0.023790
age               0.022760
capital-gain      0.021365
occupation        0.017416
hours-per-week    0.013257
capital-loss      0.004812
workclass         0.002976
sex               0.002126
native-country    0.000574
race              0.000355
dtype: float64

Feature ranking (most to least important):
['education-num', 'marital-status', 'relationship', 'age', 'capital-gain', 'occupation', 'hours-per-week', 'capital-loss', 'workclass', 'sex', 'native-country', 'race']


**Export Results**

In [ ]:
# export obtained results, including feature importance ranking
new_row = {
    "Model": ["FT-Transformer"],
    "Dataset": ["Adult Income"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_ft)],
    "Accuracy Std": [np.std(final_acc_ft)],
    "AUC-ROC Mean": [np.mean(final_auc_ft)],
    "AUC-ROC Std": [np.std(final_auc_ft)],
    "Inference Time (ms/sample)": [elapsed_ft / len(test_df_ft) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_ft)) / 1024],
    "Feature Ranking (most to least important)": [ft_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_adult_income.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Adult Income  Classification       0.848193      0.000585   
1  FT-Transformer  Adult Income  Classification       0.849592      0.001697   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.902286     0.002861                    0.010661       604.207031   
1      0.908807     0.000639                    0.059741      4893.691406   

           Feature Ranking (most to least important)  
0  ['education-num', 'relationship', 'age', 'capi...  
1  [education-num, marital-status, relationship, ...  


**XGBoost**
---
For XGBoost, categorical features are one-hot encoded, since the sklearn-style `XGBClassifier` expects numeric inputs. The "?" missing values are treated as their own category and are therefore preserved during one-hot encoding.

XGBoost does not require feature scaling, so numeric features are used in their original form. Missing numerics are retained as "?" and used directly by model. 

The same train/validation/test split is used as above, and dummy columns are aligned to the training set to ensure consistent feature representation across splits.

In [ ]:
# use original (unscaled) features for XGBoost
train_df_xgb = df.loc[X_train.index].copy().reset_index(drop=True)
val_df_xgb   = df.loc[X_val.index].copy().reset_index(drop=True)
test_df_xgb  = df.loc[X_test.index].copy().reset_index(drop=True)

X_train_xgb_raw = train_df_xgb.drop(columns=["income"])
X_val_xgb_raw   = val_df_xgb.drop(columns=["income"])
X_test_xgb_raw  = test_df_xgb.drop(columns=["income"])

y_train_xgb = train_df_xgb["income"].values
y_val_xgb   = val_df_xgb["income"].values
y_test_xgb  = test_df_xgb["income"].values

# one-hot encode categoricals
X_train_xgb = pd.get_dummies(X_train_xgb_raw, columns=CATEGORICAL_COLS, drop_first=False)
X_val_xgb   = pd.get_dummies(X_val_xgb_raw, columns=CATEGORICAL_COLS, drop_first=False)
X_test_xgb  = pd.get_dummies(X_test_xgb_raw, columns=CATEGORICAL_COLS, drop_first=False)

# align validation/test columns to training columns
X_val_xgb = X_val_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:

- n_estimators: number of boosting trees in the ensemble
- max_depth: maximum depth of each decision tree
- learning_rate: step size controlling the contribution of each tree

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 0,
        "n_jobs": -1
    }

    model_xgb = XGBClassifier(**params)
    model_xgb.fit(X_train_xgb, y_train_xgb)

    preds_proba = model_xgb.predict_proba(X_val_xgb)[:, 1]
    auc = roc_auc_score(y_val_xgb, preds_proba)

    return auc

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("Best AUC:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

Best AUC: 0.9319091799166509
Best params: {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.040725246495859874}


**Final Evaluation**

In [ ]:
best_params_xgb = study_xgb.best_params
seeds = [1, 2, 3]

final_acc_xgb, final_auc_xgb = [], []

for seed in seeds:
    model_xgb = XGBClassifier(
        **best_params_xgb,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=seed,
        n_jobs=-1
    )

    model_xgb.fit(X_train_xgb, y_train_xgb)

    preds = model_xgb.predict(X_test_xgb)
    preds_proba = model_xgb.predict_proba(X_test_xgb)[:, 1]

    final_acc_xgb.append(accuracy_score(y_test_xgb, preds))
    final_auc_xgb.append(roc_auc_score(y_test_xgb, preds_proba))

    print(f"seed={seed}, acc={final_acc_xgb[-1]:.10f}, auc={final_auc_xgb[-1]:.10f}")

print(final_acc_xgb)
print(final_auc_xgb)
print(f"XGBoost Accuracy: {np.mean(final_acc_xgb):.4f} ± {np.std(final_acc_xgb):.4f}")
print(f"XGBoost AUC-ROC:  {np.mean(final_auc_xgb):.4f} ± {np.std(final_auc_xgb):.4f}")

seed=1, acc=0.8753198894, auc=0.9298269773
seed=2, acc=0.8752175248, auc=0.9295246810
seed=3, acc=0.8758317126, auc=0.9299531164
[0.8753198894462074, 0.8752175248234211, 0.8758317125601393]
[0.9298269773389377, 0.9295246809570201, 0.9299531164328014]
XGBoost Accuracy: 0.8755 ± 0.0003
XGBoost AUC-ROC:  0.9298 ± 0.0002


**Inference Cost**

In [ ]:
start = time.time()
_ = model_xgb.predict(X_test_xgb)
elapsed_xgb = time.time() - start

print(f"Inference time per sample: {elapsed_xgb / len(X_test_xgb) * 1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_xgb)) / 1024:.2f} KB")

Inference time per sample: 0.0013 ms
Model size: 1422.14 KB


**Feature Importance**

In [ ]:
feature_importance_xgb = (
    pd.Series(model_xgb.feature_importances_, index=X_train_xgb.columns)
    .sort_values(ascending=False)
)

xgb_feature_ranking = feature_importance_xgb.index.tolist()

print(feature_importance_xgb.head(20))
print(xgb_feature_ranking)

marital-status_Married-civ-spouse    0.391248
relationship_Husband                 0.056685
capital-gain                         0.054968
education-num                        0.028818
relationship_Wife                    0.020990
capital-loss                         0.018644
marital-status_Never-married         0.017152
occupation_Farming-fishing           0.014455
relationship_Own-child               0.013918
occupation_Exec-managerial           0.013273
occupation_Handlers-cleaners         0.012444
occupation_Other-service             0.012064
occupation_Prof-specialty            0.011746
native-country_Mexico                0.010596
occupation_Tech-support              0.010231
relationship_Other-relative          0.008580
marital-status_Married-AF-spouse     0.008577
workclass_Federal-gov                0.008501
hours-per-week                       0.008264
marital-status_Divorced              0.008228
dtype: float32
['marital-status_Married-civ-spouse', 'relationship_Husband', 'ca

**Export Results**

In [ ]:
results_df = results_df.drop(columns=["AUC Mean", "AUC Std"], errors="ignore")

new_row = {
    "Model": ["XGBoost"],
    "Dataset": ["Adult Income"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_xgb)],
    "Accuracy Std": [np.std(final_acc_xgb)],
    "AUC-ROC Mean": [np.mean(final_auc_xgb)],
    "AUC-ROC Std": [np.std(final_auc_xgb)],
    "Inference Time (ms/sample)": [elapsed_xgb / len(X_test_xgb) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_xgb)) / 1024],
    "Feature Ranking (most to least important)": [xgb_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_adult_income.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Adult Income  Classification       0.848193      0.000585   
1  FT-Transformer  Adult Income  Classification       0.849592      0.001697   
2         XGBoost  Adult Income  Classification       0.875456      0.000269   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.902286     0.002861                    0.010661       604.207031   
1      0.908807     0.000639                    0.059741      4893.691406   
2      0.929768     0.000180                    0.001343      1422.136719   

           Feature Ranking (most to least important)  
0  ['education-num', 'relationship', 'age', 'capi...  
1  [education-num, marital-status, relationship, ...  
2  [marital-status_Married-civ-spouse, relationsh...  


**Random Forest**
---
For Random Forest, categorical features are one-hot encoded, since Random Forest expects numeric inputs. The "?" missing values are treated as their own category and are therefore preserved during one-hot encoding.

Random Forest does not require feature scaling, so numeric features are used in their original form. Missing numerics are retained as "?" and used directly by model. 

The same train/validation/test split is used as above, and dummy columns are aligned to the training set to ensure consistent feature representation across splits.

In [ ]:
X_train_rf = X_train_xgb.copy()
X_val_rf   = X_val_xgb.copy()
X_test_rf  = X_test_xgb.copy()

y_train_rf = y_train_xgb.copy()
y_val_rf   = y_val_xgb.copy()
y_test_rf  = y_test_xgb.copy()

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:

- `n_estimators`: number of trees in the forest  
- `max_depth`: maximum depth of each tree  
- `max_features`: number of features considered at each split  

Validation AUC is used as the optimization objective.

In [ ]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 0,
        "n_jobs": -1
    }

    model_rf = RandomForestClassifier(**params)
    model_rf.fit(X_train_rf, y_train_rf)

    preds_proba = model_rf.predict_proba(X_val_rf)[:, 1]
    auc = roc_auc_score(y_val_rf, preds_proba)

    return auc

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=50)

print("Best AUC:", study_rf.best_value)
print("Best params:", study_rf.best_params)

Best AUC: 0.9203886136135229
Best params: {'n_estimators': 300, 'max_depth': 18, 'max_features': 'sqrt'}


**Final Evaluation**

In [ ]:
best_params_rf = study_rf.best_params
seeds = [1, 2, 3]

final_acc_rf, final_auc_rf = [], []

for seed in seeds:
    model_rf = RandomForestClassifier(
        **best_params_rf,
        random_state=seed,
        n_jobs=-1
    )

    model_rf.fit(X_train_rf, y_train_rf)

    preds = model_rf.predict(X_test_rf)
    preds_proba = model_rf.predict_proba(X_test_rf)[:, 1]

    final_acc_rf.append(accuracy_score(y_test_rf, preds))
    final_auc_rf.append(roc_auc_score(y_test_rf, preds_proba))

    print(f"seed={seed}, acc={final_acc_rf[-1]:.10f}, auc={final_auc_rf[-1]:.10f}")

print(f"RF Accuracy: {np.mean(final_acc_rf):.4f} ± {np.std(final_acc_rf):.4f}")
print(f"RF AUC-ROC:  {np.mean(final_auc_rf):.4f} ± {np.std(final_auc_rf):.4f}")

seed=1, acc=0.8636503224, auc=0.9184523565
seed=2, acc=0.8638550517, auc=0.9184779124
seed=3, acc=0.8634455932, auc=0.9178651176
RF Accuracy: 0.8637 ± 0.0002
RF AUC-ROC:  0.9183 ± 0.0003


**Inference Cost**

In [ ]:
start = time.time()
_ = model_rf.predict(X_test_rf)
elapsed_rf = time.time() - start

print(f"Inference time per sample: {elapsed_rf / len(X_test_rf) * 1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_rf)) / 1024:.2f} KB")

Inference time per sample: 0.0074 ms
Model size: 69793.12 KB


**Feature Importance**

In [ ]:
rf_feature_importance = (
    pd.Series(model_rf.feature_importances_, index=X_train_rf.columns)
    .sort_values(ascending=False)
)

rf_feature_ranking = rf_feature_importance.index.tolist()

print(rf_feature_importance.head(20))
print(rf_feature_ranking)

capital-gain                         0.153518
education-num                        0.134236
marital-status_Married-civ-spouse    0.104601
age                                  0.100705
hours-per-week                       0.076387
relationship_Husband                 0.071048
capital-loss                         0.045168
marital-status_Never-married         0.038946
occupation_Exec-managerial           0.027797
occupation_Prof-specialty            0.024259
relationship_Not-in-family           0.017991
relationship_Wife                    0.016783
sex_Female                           0.014767
sex_Male                             0.014534
relationship_Own-child               0.012523
marital-status_Divorced              0.010006
occupation_Other-service             0.008130
relationship_Unmarried               0.007523
workclass_Self-emp-not-inc           0.007466
workclass_Private                    0.007197
dtype: float64
['capital-gain', 'education-num', 'marital-status_Married-civ-spo

**Export Results**

In [ ]:
new_row = {
    "Model": ["Random Forest"],
    "Dataset": ["Adult Income"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_rf)],
    "Accuracy Std": [np.std(final_acc_rf)],
    "AUC-ROC Mean": [np.mean(final_auc_rf)],
    "AUC-ROC Std": [np.std(final_auc_rf)],
    "Inference Time (ms/sample)": [elapsed_rf / len(X_test_rf) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_rf)) / 1024],
    "Feature Ranking (most to least important)": [rf_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_adult_income.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Adult Income  Classification       0.848193      0.000585   
1  FT-Transformer  Adult Income  Classification       0.849592      0.001697   
2         XGBoost  Adult Income  Classification       0.875456      0.000269   
3   Random Forest  Adult Income  Classification       0.863650      0.000167   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.902286     0.002861                    0.010661       604.207031   
1      0.908807     0.000639                    0.059741      4893.691406   
2      0.929768     0.000180                    0.001343      1422.136719   
3      0.918265     0.000283                    0.007385     69793.124023   

           Feature Ranking (most to least important)  
0  ['education-num', 'relationship', 'age', 'capi...  
1  [education-num, marital-status, relationship, ...  
2  [marital-status_Married-civ-spouse, relations